**I choose Lane 2: Refresh / Content Opportunity Scoring.**

**ML task type:** Classification (with ranking interpretation).

**Why classification?** I am predicting a binary label: whether a page will experience a meaningful decline or opportunity over a future window. This is a natural classification problem. The model outputs a probability that a page needs review. I then *rank* pages by that probability to produce a review queue. So while the final deliverable is a ranking, the underlying model is a classifier. This is a common and defensible framing for prioritisation tasks.

**Why not regression?** Predicting exact future metrics (e.g., exact impressions change) is noisier and harder to calibrate. A binary label (e.g., “decline >20%”) is more robust and directly matches the action: review or not. Classification also allows me to use evaluation metrics like precision@K that directly measure the quality of the ranked queue.

**Target:** A binary label that indicates whether a page will decline in traffic or engagement over the next 30 days.

**Definition:** I will compute this label from the warehouse daily facts (`fact_content_daily_performance`). For each content item, I will measure its total impressions or sessions in the 30‑day window immediately following a feature‑extraction point (e.g., the end of the prior 90‑day window). If that value drops by more than a chosen threshold (e.g., 20%) relative to the prior 90‑day average, I label it as `need_refresh = 1`; otherwise `0`.

**Why this label?** It is an **observed future outcome**, not a product decision. It gives the model a clear, objective signal to learn from. The threshold (e.g., 20%) and the window length are choices I will justify in my data contract. The label is leakage‑safe because the feature window (prior 90 days) and the target window (next 30 days) do not overlap.

**Proxy?** If I were working only with the starter dataset (which lacks future windows), I could use `trend_direction == "down"` as a proxy label, but that is a current‑state bucket and not truly future‑looking. My primary plan is to use the full warehouse to build the future‑looking label. For this notebook, I show the structure using the starter data as a placeholder.

In [2]:
import os
import pandas as pd
import subprocess

# === Helper: ensure repo is cloned and return CSV path ===
def ensure_repo_and_csv():
    os.chdir('/content')
    repo_name = 'Flyrank'
    csv_relative = 'data/raw/content_refresh_anonymized.csv'
    repo_path = f'/content/{repo_name}'
    csv_path = f'{repo_path}/{csv_relative}'
    if not os.path.exists(csv_path):
        print(f"📁 Cloning repo from https://github.com/ahmad-rind/{repo_name}.git ...")
        subprocess.run(['git', 'clone', f'https://github.com/ahmad-rind/{repo_name}.git'], check=True)
        # After clone, verify
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f"Could not find {csv_path} after cloning.")
    return csv_path

# Get path and load data
csv_path = ensure_repo_and_csv()
df = pd.read_csv(csv_path)

# Show a sample of the data and a placeholder target column
df_sample = df.head(10).copy()
df_sample['target_proxy'] = (df_sample['trend_direction'] == 'down').astype(int)

print("Sample of the data with a proxy target column (future label will be built from warehouse):")
print(df_sample[['content_id', 'impressions_90d', 'trend_direction', 'target_proxy']])
print(f"\nShape of full dataset: {df.shape}")
print(f"Number of rows with proxy target=1: {df_sample['target_proxy'].sum()} out of {len(df_sample)} in sample.")

📁 Cloning repo from https://github.com/ahmad-rind/Flyrank.git ...
Sample of the data with a proxy target column (future label will be built from warehouse):
             content_id  impressions_90d trend_direction  target_proxy
0  content_304f48230142             3803            down             1
1  content_a1fb4e703a9e            15320            down             1
2  content_9aa793d4d895            12581            down             1
3  content_331d6c4de07b            11751          stable             0
4  content_d99b7a2d90ca            19140            down             1
5  content_d4084a4bc775             3970            down             1
6  content_9a34b442b552               20            down             1
7  content_a63219c6e95a             1724          stable             0
8  content_5e6c160719bc            32574            down             1
9  content_c27558df2b0c             1240            down             1

Shape of full dataset: (30000, 44)
Number of rows with proxy 

**Primary metric:** Precision@K, specifically Precision@50.

**Why Precision@50?** The business problem is about a reviewer with limited capacity — they can realistically inspect 50 pages per week (or per cycle). The metric that matters is: out of the top 50 pages the model recommends, how many actually turn out to be true positives (i.e., actually need a refresh)? This directly measures the value of the queue.

**Secondary metrics:** I will also report Average Precision (AP) to summarise the entire ranking, and Recall@K to understand how many of all true positives are captured. However, Precision@K is the primary metric because it matches the real action.

**Justification:** A model could have high overall accuracy but poor precision at the top. Precision@K forces the model to rank the most promising candidates first, which is exactly what the reviewer needs. The baseline rule (from the starter pipeline) has Precision@50 = 0.240; my goal is to beat that with a learned model.

In [3]:
# This cell just prints a reminder of the metric; no extra computation needed.
print("Precision@K is the success metric.")
print("For example, if the model recommends 50 pages and 40 are actually declining, Precision@50 = 0.80.")
print("The starter baseline got Precision@50 = 0.240; we aim to beat that.")

Precision@K is the success metric.
For example, if the model recommends 50 pages and 40 are actually declining, Precision@50 = 0.80.
The starter baseline got Precision@50 = 0.240; we aim to beat that.


**Unit of analysis:** One row = one content item (page) at a specific point in time, with aggregated features over the prior 90 days. In the final project, each row will also have a future‑window label (decline or not) derived from the daily performance table.

**Why this grain?** The decision is about which pages to review. Each page is an independent candidate. Aggregating over 90 days gives a stable snapshot of recent performance, and future movement is measured over the next 30 days. This temporal structure fits the refresh workflow perfectly.

**In the starter dataset:** The CSV is already at this grain — one row per content_id with 90‑day aggregates. Below I load it and display the first few rows to show the unit of analysis.

In [4]:
import pandas as pd
import os

os.chdir('/content')
csv_path = '/content/Flyrank/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(csv_path)

print("Unit of analysis: one row = one content page (content_id).")
print(f"Total number of pages in this slice: {len(df):,}")
print("\nFirst 5 rows (showing key columns):")
display(df.head(5))
print("\nColumn names (some are features, some are context):")
print(df.columns.tolist())

Unit of analysis: one row = one content page (content_id).
Total number of pages in this slice: 30,000

First 5 rows (showing key columns):


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7



Column names (some are features, some are context):
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


**Why a fixed rule is insufficient:**

1. **Signal interactions:** A page’s health depends on multiple factors — impressions, position, CTR, trend, engagement, age — that interact in non‑linear ways. For example, a page with high impressions but low CTR may be more urgent than a page with low impressions and declining trend, but a simple linear rule would treat them independently. A model can learn these interactions from data.

2. **Threshold selection:** The baseline rule uses hand‑picked thresholds (e.g., >=500 impressions, age >=180 days, CTR <0.5). These thresholds are arbitrary and may not be optimal for every client or content type. A model learns the optimal separation from the data, adjusting for the actual relationship between signals and future outcomes.

3. **Adaptability:** As the search landscape changes (e.g., AI‑influenced click patterns), the relationships between signals and future decline may shift. A model can be retrained on fresh data, while a fixed rule would become stale.

4. **Empirical evidence:** The starter pipeline already proves that a random forest (AUC 0.750, Precision@50 0.740) beats the baseline rule (AUC 0.627, Precision@50 0.240) on the proxy label. Even with a more rigorous future‑looking label, I expect a learned ranking to outperform a static rule because the pattern is not a simple formula.

**ML is appropriate because** the relationship between the observable signals and the future outcome is complex, data‑driven, and benefits from optimisation against the specific business metric (Precision@K).

In [5]:
# This cell just prints a summary of the baseline vs model performance from the starter pipeline.
print("Starter baseline Precision@50: 0.240")
print("Random forest Precision@50: 0.740")
print("The model more than triples the precision, showing ML can beat a fixed rule.")
print("This is empirical evidence that the problem is suited for ML.")

Starter baseline Precision@50: 0.240
Random forest Precision@50: 0.740
The model more than triples the precision, showing ML can beat a fixed rule.
This is empirical evidence that the problem is suited for ML.


**Before you submit, confirm each line honestly:**

- Every section above is filled — markdown thinking AND the code that backs it.
- The notebook runs top to bottom with no errors (Runtime → Run all).
- No client names, URLs, or private queries anywhere.
- My claims use careful words: observed, measured, directional, decision‑support.
- Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.